# **PRACTICE 5: Probabilistic Language Model**

There are aggressive and non-aggressive tweets in the training and validation files, as well as their labels. Here, an example of classification is given.

## **Repeat the first part of the last practice:**

In [1]:
# Libraries.       
import nltk                                    # library for text processing.
from nltk.tokenize import TweetTokenizer       # library for tweet tokenization.
from pathlib import Path                       # library for file path.
nltk.download("stopwords")                     # download stopwords.
from nltk.corpus import stopwords              # library for stopwords.
stopwords_es = set(stopwords.words('spanish')) # set of spanish stopwords.
import numpy as np                             # library for numerical operations.

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ezautorres/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## **Read the data and preprocessing:**

In [2]:
# Function to read the text and labels from the files.
def get_text_from_file(corpus_path: Path, labels_path: Path) -> tuple[list[str]]:
    """
    Read the text and labels from the files.

    Parameters
    ----------
    corpus_path : Path
        The path to the file containing the text.
    labels_path : Path
        The path to the file containing the labels.

    Returns
    -------
    tuple[list[str]]
        The text and labels.
    """
    with open(corpus_path, "r") as corpus_file, open(labels_path, "r") as labels_file:
        corpus = corpus_file.readlines()                 # Read the text line by line.
        labels = list(map(int, labels_file.readlines())) # Read the labels line by line.

    return corpus, labels

# Load the data.
tr_corpus, tr_labels = get_text_from_file('./mex20_train.txt', './mex20_train_labels.txt') # Training data.
val_corpus, val_labels = get_text_from_file('./mex20_val.txt', './mex20_val_labels.txt')   # Validation data.

print("Number of tweets in the training and validation set:")
print("    -   Training  :", len(tr_corpus))
print("    -   Validation:", len(val_corpus))

Number of tweets in the training and validation set:
    -   Training  : 5278
    -   Validation: 587


In [3]:
class TrigramData:
    """
    Class to preprocess the data for the trigram language model.
    """
    def __init__(self, max_vocab_size: int, tokenizer: TweetTokenizer, remove_stopwords: bool = False):
        """
        Initialize the TrigramData class.

        Parameters
        ----------
        max_vocab_size : int
            The maximum size of the vocabulary.
        tokenizer : TweetTokenizer
            The tokenizer to be used.
        remove_stopwords : bool, optional
            Whether to remove stopwords, by default False.
        """
        self.max_vocab_size = max_vocab_size     # Maximum vocabulary size.
        self.tokenizer = tokenizer               # Tokenizer.
        self.remove_stopwords = remove_stopwords # Whether to remove stopwords.
        self.UNK = "<unk>"                       # Unknown token.
        self.SOS = "<s>"                         # Start of sequence token.
        self.EOS = "</s>"                        # End of sequence token.
        self.vocab = set()                       # Final vocabulary.

    def fit(self, raw_txt: list[str]) -> list[list[str]]:
        """
        Fit the data to the model.
        
        Parameters
        ----------
        raw_txt : list[str]
            The raw text data.
        
        Returns
        -------
        list[list[str]]
            The transformed corpus.
        """
        freq_dist = nltk.FreqDist() # Frequency distribution.
        tokenized_corpus = []       # Tokenized corpus.

        for doc in raw_txt:                                                         # For each document.
            tokens = self.tokenizer.tokenize(doc)                                   # Tokenize the document.
            if self.remove_stopwords:                                               # If stopwords are to be removed.
                tokens = [w for w in tokens if w not in stopwords.words('spanish')] # Remove stopwords.
            tokenized_corpus.append(tokens)                                         # Append the tokenized document to the corpus.
            freq_dist.update(tokens)                                                # Update the frequency distribution.                    

        self.vocab = {token for token, _ in freq_dist.most_common(self.max_vocab_size)} # Create the final vocabulary.
        self.vocab.update({self.UNK, self.SOS, self.EOS})                               # Add the unknown, start of sequence, and end of sequence tokens.

        transformed_corpus = [self.transform(doc) for doc in tokenized_corpus] # Transform the corpus.
        return transformed_corpus

    def mask_oov(self, w: str) -> str:
        """
        Mask the word if it is out of vocabulary.

        Parameters
        ----------
        w : str
            The word.

        Returns
        -------
        str
            The masked word.
        """
        return w if w in self.vocab else self.UNK

    def add_sos_eos(self, tokens: list[str]) -> list[str]:
        """
        Add the start of sequence and end of sequence tokens.

        Parameters
        ----------
        tokens : list[str]
            The tokens.

        Returns
        -------
        list[str]
            The tokens with the start of sequence and end of sequence tokens.
        """
        return [self.SOS, self.SOS] + tokens + [self.EOS]

    def transform(self, tokens: list[str]) -> list[str]:
        """
        Transform the tokens. Mask the out of vocabulary words and add the start of sequence and end of sequence tokens.
     
        Parameters
        ----------
        tokens : list[str]
            The tokens.
           
        Returns
        -------
        list[str]
            The transformed tokens.
        """
        transformed = [self.mask_oov(token) for token in tokens]
        return self.add_sos_eos(transformed)

class TrigramLanguageModel:
    """
    Class to create a trigram language model.
    """
    def __init__(self, lambdas: tuple[float, float, float] = (0.4, 0.3, 0.3)):
        """
        Initialize the TrigramLanguageModel class.

        Parameters
        ----------
        lambdas : tuple[float, float, float], optional
            The lambdas for the interpolation, by default (0.4, 0.3, 0.3).
        """
        self.lambda1, self.lambda2, self.lambda3 = lambdas # Lambdas for the interpolation.
        
        # Counters.
        self.unigram_count = {} # Unigram count.
        self.bigram_count = {}  # Bigram count.
        self.trigram_count = {} # Trigram count.

        # Inicialize the vocabulary and the total number of tokens.
        self.vocab = set()      # Vocabulary.
        self.total_tokens = 0   # Total number of tokens.
        self.vocabularySize = 0 # Vocabulary size.

    def train(self, transformed_corpus: list[list[str]], vocab: set[str]):
        """
        Train the trigram language model.

        Parameters
        ----------
        transformed_corpus : list[list[str]]
            The transformed corpus.
        vocab : set[str]
            The final vocabulary.
        """
        self.vocab = vocab               # Vocabulary.
        self.vocabularySize = len(vocab) # Vocabulary size.

        for doc in transformed_corpus:  # For each document.
            for i, w in enumerate(doc): # For each word in the document.

                # Unigram.
                self.unigram_count[w] = self.unigram_count.get(w, 0) + 1

                # Bigram.
                if i > 0:
                    w_prev = doc[i-1]
                    self.bigram_count[(w_prev, w)] = self.bigram_count.get((w_prev, w), 0) + 1

                # Trigram.
                if i > 1:
                    w_prev_prev = doc[i-2]
                    self.trigram_count[(w_prev_prev, w_prev, w)] = self.trigram_count.get((w_prev_prev, w_prev, w), 0) + 1

        # Total number of tokens.
        self.total_tokens = sum(self.unigram_count.values())

    def mask_oov(self, w: str) -> str:
        """
        Mask the word if it is out of vocabulary."

        Parameters
        ----------
        w : str
            The word.
        
        Returns
        -------
        str
            The masked word.
        """
        return w if w in self.vocab else "<UNK>"

    def unigram_probability(self, w: str) -> float:
        """
        Calculate the unigram probability.

        Parameters
        ----------
        w : str
            The word.

        Returns
        -------
        float
            The unigram probability.
        """
        return (self.unigram_count.get(self.mask_oov(w), 0) + 1) / (self.total_tokens + self.vocabularySize)

    def bigram_probability(self, w_prev: str, w: str) -> float:
        """
        Calculate the bigram probability.

        Parameters
        ----------
        w_prev : str
            The previous word.
        w : str
            The word.

        Returns
        -------
        float
            The bigram probability.
        """
        return (self.bigram_count.get((self.mask_oov(w_prev), self.mask_oov(w)), 0) + 1) / (self.unigram_count.get(self.mask_oov(w_prev), 0) + self.vocabularySize)

    def trigram_probability(self, w_prev_prev: str, w_prev: str, w: str) -> float:
        """
        Calculate the trigram probability."
        ""
        Parameters
        ----------
        w_prev_prev : str
            The previous previous word.
        w_prev : str
            The previous word.
        w : str
            The word.
            
        Returns
        -------
        float
            The trigram probability.
        """
        return (self.trigram_count.get((self.mask_oov(w_prev_prev), self.mask_oov(w_prev), self.mask_oov(w)), 0) + 1) / (self.bigram_count.get((self.mask_oov(w_prev_prev), self.mask_oov(w_prev)), 0) + self.vocabularySize)

    def word_probability(self, w_prev_prev: str, w_prev: str, w: str) -> float:
        """
        Calculate the word probability.

        Parameters
        ----------
        w_prev_prev : str
            The previous previous word.
        w_prev : str
            The previous word.
        w : str
            The word.

        Returns
        -------
        float
            The word probability.
        """
        return (self.lambda1 * self.unigram_probability(w) +
                self.lambda2 * self.bigram_probability(w_prev, w) +
                self.lambda3 * self.trigram_probability(w_prev_prev, w_prev, w))

    def sentence_probability(self, sequence: list[str]) -> float:
        """
        Calculate the probability of a sentence.
      
        Parameters
        ----------
        sequence : list[str]
            The sequence.
        
        Returns
        -------
        float
            The probability of the sentence.
        """
        log_prob = 0                      # Log probability.
        for i in range(2, len(sequence)): # For each word in the sequence.
            w_prev_prev = sequence[i-2]   # Previous previous word.
            w_prev = sequence[i-1]        # Previous word.
            w = sequence[i]               # Word.
            log_prob += np.log(self.word_probability(w_prev_prev, w_prev, w))
        return np.exp(log_prob)

    def check_probs(self):
        print(sum(self.unigram_probability(w) for w in self.vocab))
        print(sum(self.bigram_probability("hola", w) for w in self.vocab))
        print(sum(self.trigram_probability("hola", "como", w) for w in self.vocab))

We tokenize with TweetTokenizer. As parameters in the tokenizer we specify that the text should be converted to lowercase (```preserve_case = False```, that the words "@user" should be removed (```strip_handles = True```), and that sequences with more than 3 identical characters in a row should be cut to length 3 (```reduce_len = True```).

In [4]:
# Tokenizer.
tokenizer = TweetTokenizer(preserve_case = False, reduce_len = True, strip_handles = True)

# Preprocessing.
trigram_data = TrigramData(max_vocab_size = 13071, tokenizer = tokenizer, remove_stopwords = False) # Initialize the trigram data.
transformed_corpus = trigram_data.fit(tr_corpus)                                                   # Fit the data.

# Training.
vocab = trigram_data.vocab                                   # Vocabulary.
trigram_LM = TrigramLanguageModel(lambdas = (0.6, 0.3, 0.1)) # Initialize the trigram language model.
trigram_LM.train(transformed_corpus, vocab)                  # Train the model.

# Check probabilities.
trigram_LM.check_probs()

0.999999999999909
1.0000000000002276
0.9999999999999098


Test

In [5]:
w_prev_prev, w_prev, w = "<s>", "hola", "mundo"
p_w = trigram_LM.word_probability(w_prev_prev, w_prev, w)
print(f"p({w}|{w_prev_prev},{w_prev}) = {p_w}")

p(mundo|<s>,hola) = 0.0002946371793914035


In [6]:
w_prev_prev, w_prev, w = "<s>", "saludos", "mundo"
p_w = trigram_LM.word_probability(w_prev_prev, w_prev, w)
print(f"p({w}|{w_prev_prev},{w_prev}) = {p_w}")

p(mundo|<s>,saludos) = 0.00029464302475021574


In [7]:
w_prev_prev, w_prev, w = "vete", "a", "la"
p_w = trigram_LM.word_probability(w_prev_prev, w_prev, w)
print(f"p({w}|{w_prev_prev},{w_prev}) = {p_w}")

p(la|vete,a) = 0.01716772192700376


In [8]:
seq = ["hola", "como", "has", "estado", "<\s>"]
p_seq = trigram_LM.sentence_probability(seq)
print(f"Probabilidad de la secuencia '{' '.join(seq)}': {p_seq}")

Probabilidad de la secuencia 'hola como has estado <\s>': 3.518002489834672e-13
